# Edge check: crm-prototype-client (JavaScript / JSX)
Expected edges come from the **TypeScript compiler (`js-ref/edges.mjs`, `functions.mjs`, `classes.mjs`)**, not tree-sitter. They are compared with code-review-graph (CRG) `graph.db`, opened read-only, and with the tree-sitter module (`core-module/edges.py`).

| Edge | Reference | Match key |
|---|---|---|
| CONTAINS | named functions and classes (the same references as the function and class checks) | file + start line + name |
| INHERITS | `class X extends Y` | file + class start line + parent name |
| IMPORTS_FROM | `import … from`, `export … from`, `require(...)`, `import(...)` | file + line |
| CALLS | `f()`, `a.b()`, `new X()`, JSX `<Foo/>` (it compiles to `React.createElement(Foo)`) | file + line + name (line of the name, else line where the call starts) |

- **Recall** = expected edges found. **Precision** = reported edges that the reference also has.
- CALLS = call sites by name. It doesn't check which function is reached.
- REFERENCES edges (CRG has them here) are not checked yet.
- Each run is saved to `results/crm-prototype-client/checks/<run>/` and logged in `results/crm-prototype-client/checks.csv`.

In [1]:
import json, sqlite3, subprocess, pathlib, datetime, sys
import pandas as pd
pd.set_option('display.max_colwidth', 90)
ROOT = pathlib.Path('..').resolve()
APP, LANG = 'crm-prototype-client', 'javascript'
REPO = ROOT / 'apps' / APP
CRG_RUN = 'iter-006_2026-11-18_0744'
DB = ROOT / f'results/{APP}/runs/{CRG_RUN}/crg/graph.db'
RUN = 'edges_' + datetime.datetime.now().strftime('%Y-%m-%d_%H%M')
OUT = ROOT / f'results/{APP}/checks/{RUN}'; OUT.mkdir(parents=True, exist_ok=True)
REF_COMMIT = subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip()
EXT = {'javascript': ['*.js','*.jsx','*.mjs','*.cjs'], 'python': ['*.py']}[LANG]
files = subprocess.run(['git','ls-files', *EXT], cwd=REPO, capture_output=True, text=True, check=True).stdout.split()
print(RUN, '| repo commit', REF_COMMIT, '| CRG run', CRG_RUN, '| files', len(files))

edges_2026-11-20_1245 | repo commit 325788b | CRG run iter-006_2026-11-18_0744 | files 157


## 1. Reference

In [2]:
out = subprocess.run(['node', str(ROOT/'js-ref/edges.mjs'), str(REPO)], input='\n'.join(files), capture_output=True, text=True, check=True).stdout
rows = [json.loads(l) for l in out.splitlines()]
fn = json.loads(subprocess.run(['node', str(ROOT/'js-ref/functions.mjs'), str(REPO), *files], capture_output=True, text=True, check=True).stdout)
rows += [dict(e='contains', file=f, line=r['line'], child=r['name'], kind='function') for f, d in fn.items() for r in d.get('named', [])]
cl = subprocess.run(['node', str(ROOT/'js-ref/classes.mjs'), str(REPO)], input='\n'.join(files), capture_output=True, text=True, check=True).stdout
rows += [dict(e='contains', file=r['file'], line=r['line'], child=r['name'], kind=r['kind']) for r in map(json.loads, cl.splitlines()) if r['name']]
errors = sum('error' in d for d in fn.values())

In [3]:
(OUT/'ref_edges.jsonl').write_text('\n'.join(json.dumps(r) for r in rows))
KEY = {'CONTAINS':'contains','INHERITS':'inherits','IMPORTS_FROM':'import','CALLS':'call'}
R = {e: pd.DataFrame([r for r in rows if r['e'] == k], columns=None) for e, k in KEY.items()}
for e in R:
    if R[e].empty: R[e] = pd.DataFrame(columns=['file','line','child','sup','name','start','def_line'])
print('parse errors:', errors)
pd.Series({e: len(d) for e, d in R.items()}, name='reference rows')

parse errors: 0


CONTAINS         624
INHERITS           0
IMPORTS_FROM     833
CALLS           6312
Name: reference rows, dtype: int64

## 2. code-review-graph edges

In [4]:
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
P = str(REPO) + '/'
rel = lambda s: s.str.replace(P, '', regex=False)
nodes = pd.read_sql('select qualified_name, name, line_start, kind node_kind from nodes', con); nodes['qualified_name'] = rel(nodes.qualified_name)
def crg(kind):
    d = pd.read_sql('select source_qualified src, target_qualified tgt, file_path file, line from edges where kind=?', con, params=(kind,))
    for c in ['src','tgt','file']: d[c] = rel(d[c])
    return d[d.file.isin(set(files))].reset_index(drop=True)
last = lambda t: t.split('::')[-1].split('.')[-1].split(':')[0]
c = crg('CONTAINS').merge(nodes, left_on='tgt', right_on='qualified_name', how='left')
c = c[c.node_kind.isin(['Function','Test','Class'])].rename(columns={'line_start':'child_line','name':'child'}).reset_index(drop=True)
i = crg('INHERITS').merge(nodes[['qualified_name','line_start']], left_on='src', right_on='qualified_name', how='left'); i['sup'] = i.tgt.map(last)
k = crg('CALLS'); k['name'] = k.tgt.map(last)
G = {'CONTAINS': c, 'INHERITS': i, 'IMPORTS_FROM': crg('IMPORTS_FROM'), 'CALLS': k}
print('CRG REFERENCES edges in these files (not checked):', len(crg('REFERENCES')))
pd.Series({e: len(d) for e, d in G.items()}, name='CRG rows (code files; CONTAINS -> functions/classes)')

CRG REFERENCES edges in these files (not checked): 22


CONTAINS         564
INHERITS           0
IMPORTS_FROM     792
CALLS           5213
Name: CRG rows (code files; CONTAINS -> functions/classes), dtype: int64

Matching is multiset: each edge is used at most once. A second key is tried only for rows still unmatched.

In [5]:
def match(ref, got, passes):
    ref = ref.copy(); got = got.copy(); ref['hit'] = False; got['hit'] = False
    for rk, gk in passes:
        if ref.empty or got.empty: break
        rr = ref[~ref.hit].copy(); gg = got[~got.hit].copy()
        rr['_n'] = rr.groupby(rk).cumcount(); gg['_n'] = gg.groupby(gk).cumcount()
        j = rr.reset_index().merge(gg.reset_index(), left_on=rk+['_n'], right_on=gk+['_n'], suffixes=('','_g'))
        ref.loc[j['index'], 'hit'] = True; got.loc[j['index_g'], 'hit'] = True
    return ref, got
SUMMARY, RES = [], {}
def report(label, ref, got):
    m, e = int(ref.hit.sum()), int(got.hit.sum())
    SUMMARY.append({'edge': label, 'expected': len(ref), 'reported': len(got), 'matched': m, 'missed': len(ref)-m,
                    'extra': len(got)-e, 'recall %': round(100*m/len(ref),1) if len(ref) else None,
                    'precision %': round(100*e/len(got),1) if len(got) else None})
    RES[label] = (ref, got)
    ref[~ref.hit].drop(columns='hit').to_csv(OUT/f'{label.replace(" ","_")}_missed.csv', index=False)
    got[~got.hit].drop(columns='hit').to_csv(OUT/f'{label.replace(" ","_")}_extra.csv', index=False)
CONT2 = []
CRG_KEYS = {'CONTAINS': [(['file','line','child'], ['file','child_line','child'])] + CONT2,
            'INHERITS': [(['file','line','sup'], ['file','line_start','sup'])] + [],
            'IMPORTS_FROM': [(['file','line'], ['file','line'])],
            'CALLS': [(['file','line','name'], ['file','line','name']), (['file','start','name'], ['file','line','name'])]}

## 3. CRG vs reference

In [6]:
for e in G: report(e, *match(R[e], G[e], CRG_KEYS[e]))
pd.DataFrame(SUMMARY).set_index('edge')

,expected,reported,matched,missed,extra,recall %,precision %
edge,,,,,,,
CONTAINS,624,564,560,64,4,89.7,99.3
INHERITS,0,0,0,0,0,NaN,NaN
IMPORTS_FROM,833,792,792,41,0,95.1,100.0
CALLS,6312,5213,5213,1099,0,82.6,100.0


### Why CRG misses or adds edges

In [7]:
def why(label, col, mapping=None):
    ref, got = RES[label]
    m = ref[~ref.hit]
    return (m[col].map(mapping) if mapping else m[col]).fillna('other').value_counts().rename(f'{label} missed by {col}')
display(why('CALLS', 'how', {'call':'f() / a.b()', 'new':'new X()', 'jsx':'JSX component <Foo/>', 'jsx_html':'JSX html tag <div/>'}))
display(why('IMPORTS_FROM', 'how'))
display(why('CONTAINS', 'kind'))
display(RES['INHERITS'][0][~RES['INHERITS'][0].hit].head(10))

how
JSX html tag <div/>     976
new X()                  87
f() / a.b()              33
JSX component <Foo/>      3
Name: CALLS missed by how, dtype: int64

how
import    41
Name: IMPORTS_FROM missed by how, dtype: int64

kind
function    64
Name: CONTAINS missed by kind, dtype: int64

,file,line,child,sup,name,start,def_line,hit


In [8]:
for label in G:
    print(label, 'extra (CRG only), first rows:')
    display(RES[label][1][~RES[label][1].hit].head(5))

CONTAINS extra (CRG only), first rows:


,src,tgt,file,line,qualified_name,child,child_line,node_kind,hit
76,src/components/forms/call-logs/CallLogForm.jsx,src/components/forms/call-logs/CallLogForm.jsx::initCompanyId,src/components/forms/call-logs/CallLogForm.jsx,92,src/components/forms/call-logs/CallLogForm.jsx::initCompanyId,initCompanyId,92,Function,False
77,src/components/forms/call-logs/CallLogForm.jsx,src/components/forms/call-logs/CallLogForm.jsx::initLead,src/components/forms/call-logs/CallLogForm.jsx,101,src/components/forms/call-logs/CallLogForm.jsx::initLead,initLead,101,Function,False
503,src/store/userStore.jsx,src/store/userStore.jsx::sessionStorage,src/store/userStore.jsx,20,src/store/userStore.jsx::sessionStorage,sessionStorage,20,Function,False
510,src/utils/axios/axiosPrivateInstance.js,src/utils/axios/axiosPrivateInstance.js::res,src/utils/axios/axiosPrivateInstance.js,24,src/utils/axios/axiosPrivateInstance.js::res,res,24,Function,False


INHERITS extra (CRG only), first rows:


,src,tgt,file,line,qualified_name,line_start,sup,hit


IMPORTS_FROM extra (CRG only), first rows:


,src,tgt,file,line,hit


CALLS extra (CRG only), first rows:


,src,tgt,file,line,name,hit


## 4. Tree-sitter module (`core-module/edges.py`)
Contains comes from the module's own function finder (`extract.py`) and class finder (`classes.py`).

In [9]:
sys.path.insert(0, str(ROOT/'core-module'))
import importlib, edges as ME; importlib.reload(ME)
M = {k: [] for k in KEY.values()}
for f in files:
    for k2, v in ME.module_edges(REPO, f, LANG).items(): M[k2] += v
with open(OUT/'module_edges.jsonl','w') as fh:
    for k2, v in M.items():
        for r in v: fh.write(json.dumps({'e': k2, **r}) + '\n')
M = {e: pd.DataFrame(M[k2]) if M[k2] else pd.DataFrame(columns=R[e].columns) for e, k2 in KEY.items()}
MOD_KEYS = {'CONTAINS': [(['file','line','child'], ['file','line','child'])] + [],
            'INHERITS': [(['file','line','sup'], ['file','line','sup'])],
            'IMPORTS_FROM': [(['file','line'], ['file','line'])],
            'CALLS': [(['file','line','name'], ['file','line','name']), (['file','start','name'], ['file','start','name'])]}
for e in M: report(f'{e} (module)', *match(R[e], M[e], MOD_KEYS[e]))
S = pd.DataFrame(SUMMARY).set_index('edge')
pd.DataFrame({e: {'expected': S.loc[e,'expected'], 'CRG recall %': S.loc[e,'recall %'], 'module recall %': S.loc[f'{e} (module)','recall %'],
                  'CRG precision %': S.loc[e,'precision %'], 'module precision %': S.loc[f'{e} (module)','precision %'],
                  'CRG missed': S.loc[e,'missed'], 'module missed': S.loc[f'{e} (module)','missed'],
                  'CRG extra': S.loc[e,'extra'], 'module extra': S.loc[f'{e} (module)','extra']} for e in G}).T

,expected,CRG recall %,module recall %,CRG precision %,module precision %,CRG missed,module missed,CRG extra,module extra
CONTAINS,624.0,89.7,99.2,99.3,99.2,64.0,5.0,4.0,5.0
INHERITS,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
IMPORTS_FROM,833.0,95.1,100.0,100.0,100.0,41.0,0.0,0.0,0.0
CALLS,6312.0,82.6,100.0,100.0,100.0,1099.0,0.0,0.0,0.0


In [10]:
for e in G:
    ref, got = RES[f'{e} (module)']
    if (~ref.hit).any() or (~got.hit).any():
        print(e, '- module missed / extra:'); display(ref[~ref.hit].head(5)); display(got[~got.hit].head(5))

CONTAINS - module missed / extra:


,e,file,line,child,kind,hit
123,contains,src/components/ui/accordion.jsx,18,AccordionTrigger,function,False
124,contains,src/components/ui/accordion.jsx,37,AccordionContent,function,False
158,contains,src/components/ui/dialog.jsx,27,DialogContent,function,False
213,contains,src/components/ui/form.jsx,104,FormMessage,function,False
243,contains,src/components/ui/sidebar.jsx,61,setOpen,function,False


,file,line,child,kind,hit
123,src/components/ui/accordion.jsx,19,AccordionTrigger,function,False
124,src/components/ui/accordion.jsx,38,AccordionContent,function,False
158,src/components/ui/dialog.jsx,28,DialogContent,function,False
213,src/components/ui/form.jsx,105,FormMessage,function,False
243,src/components/ui/sidebar.jsx,62,setOpen,function,False


## 5. Save

In [11]:
S = pd.DataFrame(SUMMARY).set_index('edge'); S.to_csv(OUT/'summary.csv')
log = ROOT / f'results/{APP}/checks.csv'
row = S.reset_index().rename(columns={'expected':'expected (javac)','reported':'CRG','extra':'CRG extra'})
row = row.assign(run=RUN, repo_commit=REF_COMMIT, crg_run=CRG_RUN, reference=('TypeScript compiler (js-ref/edges.mjs)' if LANG=='javascript' else 'CPython ast (lang-ref/python/edges.py)'))
row.to_csv(log, mode='a', header=not log.exists(), index=False)
print('saved to', OUT.relative_to(ROOT), '| logged in', log.relative_to(ROOT))
S

saved to results/crm-prototype-client/checks/edges_2026-11-20_1245 | logged in results/crm-prototype-client/checks.csv


,expected,reported,matched,missed,extra,recall %,precision %
edge,,,,,,,
CONTAINS,624,564,560,64,4,89.7,99.3
INHERITS,0,0,0,0,0,NaN,NaN
IMPORTS_FROM,833,792,792,41,0,95.1,100.0
CALLS,6312,5213,5213,1099,0,82.6,100.0
CONTAINS (module),624,624,619,5,5,99.2,99.2
INHERITS (module),0,0,0,0,0,NaN,NaN
IMPORTS_FROM (module),833,833,833,0,0,100.0,100.0
CALLS (module),6312,6312,6312,0,0,100.0,100.0
